In [ ]:
"""
Project 04: Hospital Readmission
Step 03: Evaluation and Interpretability
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 03: Evaluation and Interpretability")
print("=" * 60)
start_time = time.time()

In [ ]:
# ─── 1. Load Data and Models ───
print("\n[1/4] Loading data and models...")
df = pd.read_csv(os.path.join(PROCESSED, "readmission_cleaned.csv"))
with open(os.path.join(MODELS, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)
with open(os.path.join(MODELS, "label_encoders.pkl"), "rb") as f:
    le_dict = pickle.load(f)
with open(os.path.join(MODELS, "lgbm_calibrated.pkl"), "rb") as f:
    model_calibrated = pickle.load(f)
with open(os.path.join(MODELS, "logistic_baseline.pkl"), "rb") as f:
    model_baseline = pickle.load(f)

In [ ]:
print(f"  Loaded: {df.shape[0]} records")

In [ ]:
X = df.drop(
    ["encounter_id", "patient_nbr", "readmitted", "readmitted_30d"],
    axis=1,
    errors="ignore",
)
y = df["readmitted_30d"]

In [ ]:
for col, le in tqdm(le_dict.items(), desc="  Encoding features"):
    if col in X.columns:
        X[col] = le.transform(X[col].astype(str))

In [ ]:
X_scaled = scaler.transform(X)

In [ ]:
# ─── 2. Predictions and Metrics ───
print("[2/4] Computing predictions and metrics...")
probs_lgb = model_calibrated.predict_proba(X_scaled)[:, 1]
preds_lgb = model_calibrated.predict(X_scaled)

In [ ]:
probs_lr = model_baseline.predict_proba(X_scaled)[:, 1]
preds_lr = model_baseline.predict(X_scaled)

In [ ]:
results = []
for name, probs, preds in [
    ("LightGBM Calibrated", probs_lgb, preds_lgb),
    ("Logistic Regression", probs_lr, preds_lr),
]:
    auroc = roc_auc_score(y, probs)
    pr_auc = average_precision_score(y, probs)
    brier = brier_score_loss(y, probs)
    cm = confusion_matrix(y, preds)
    report = classification_report(y, preds, output_dict=True)
    results.append(
        {
            "Model": name,
            "AUROC": auroc,
            "PR-AUC": pr_auc,
            "Brier Score": brier,
            "Accuracy": report["accuracy"],
            "Precision_1": report.get("1", {}).get("precision", 0),
            "Recall_1": report.get("1", {}).get("recall", 0),
            "F1_1": report.get("1", {}).get("f1-score", 0),
        }
    )
    print(f"\n  {name}:")
    print(f"    AUROC: {auroc:.4f} | PR-AUC: {pr_auc:.4f} | Brier: {brier:.4f}")

In [ ]:
metrics_df = pd.DataFrame(results)
metrics_df.to_csv(os.path.join(MODELS, "performance_metrics.csv"), index=False)
print(f"\n  Saved: performance_metrics.csv")

In [ ]:
# ─── 3. ROC Curve and Calibration Curve ───
print("[3/4] Generating performance charts...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

In [ ]:
for name, probs in [
    ("LightGBM Calibrated", probs_lgb),
    ("Logistic Regression", probs_lr),
]:
    fpr, tpr, _ = roc_curve(y, probs)
    auroc = roc_auc_score(y, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auroc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("ROC Curve", fontsize=14)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

In [ ]:
for name, probs in [
    ("LightGBM Calibrated", probs_lgb),
    ("Logistic Regression", probs_lr),
]:
    prob_true, prob_pred = calibration_curve(y, probs, n_bins=10)
    axes[1].plot(prob_pred, prob_true, marker="o", label=name)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect Calibration")
axes[1].set_title("Calibration Curve", fontsize=14)
axes[1].set_xlabel("Mean Predicted Probability")
axes[1].set_ylabel("Fraction of Positives")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "model_performance_curves.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"  Saved: charts/model_performance_curves.png")

In [ ]:
# Confusion Matrix Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, preds) in zip(
    axes, [("LightGBM Calibrated", preds_lgb), ("Logistic Regression", preds_lr)]
):
    cm = confusion_matrix(y, preds)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name}\nConfusion Matrix", fontsize=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Readmit", "Readmit"])
    ax.set_yticklabels(["No Readmit", "Readmit"])
    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=12,
            )
plt.colorbar(im, ax=axes, shrink=0.8)
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "confusion_matrices.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"  Saved: charts/confusion_matrices.png")

In [ ]:
# ─── 4. SHAP Analysis ───
print("[4/4] Running SHAP analysis...")
try:
    import shap

    # Get the underlying estimator from calibrated model
    if hasattr(model_calibrated, "calibrated_classifiers_"):
        base_estimator = model_calibrated.calibrated_classifiers_[0].estimator
    elif hasattr(model_calibrated, "estimator"):
        base_estimator = model_calibrated.estimator
    else:
        base_estimator = model_calibrated

    explainer = shap.TreeExplainer(base_estimator)
    sample_size = min(500, len(X_scaled))
    X_sample = X_scaled[:sample_size]
    shap_values = explainer.shap_values(X_sample)

    # Handle binary classification output
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_vals, X_sample, feature_names=X.columns, show=False, max_display=15
    )
    plt.title("Clinical Feature Drivers (SHAP)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "shap_summary.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: charts/shap_summary.png")

    # Feature importance bar chart
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(
        shap_vals,
        X_sample,
        feature_names=X.columns,
        plot_type="bar",
        show=False,
        max_display=15,
    )
    plt.title("Top 15 Feature Importance (SHAP)", fontsize=14)
    plt.tight_layout()
    plt.savefig(
        os.path.join(CHARTS, "feature_importance.png"), dpi=150, bbox_inches="tight"
    )
    plt.close()
    print(f"  Saved: charts/feature_importance.png")
except Exception as e:
    print(f"  SHAP analysis skipped: {e}")
    # Fallback: LightGBM feature importance
    try:
        if hasattr(model_calibrated, "calibrated_classifiers_"):
            base_estimator = model_calibrated.calibrated_classifiers_[0].estimator
        elif hasattr(model_calibrated, "estimator"):
            base_estimator = model_calibrated.estimator
        else:
            base_estimator = model_calibrated
        importances = base_estimator.feature_importances_
        feat_imp = (
            pd.Series(importances, index=X.columns).sort_values(ascending=True).tail(15)
        )
        fig, ax = plt.subplots(figsize=(10, 6))
        feat_imp.plot(kind="barh", ax=ax, color="#2196F3")
        ax.set_title("Top 15 Feature Importance (LightGBM)", fontsize=14)
        ax.set_xlabel("Importance")
        plt.tight_layout()
        plt.savefig(
            os.path.join(CHARTS, "feature_importance.png"), dpi=150, bbox_inches="tight"
        )
        plt.close()
        print(f"  Saved: charts/feature_importance.png (fallback)")
    except Exception as e2:
        print(f"  Fallback importance also failed: {e2}")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 03 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")